# Lesson 6 — Tokens, for Real

Byte-pair encoding in about forty lines: merge the most frequent adjacent
pair, repeat, and a vocabulary invents itself from the text.

In [ ]:
# The corpus: Alice in Wonderland (public domain), with a built-in backup.
import urllib.request, re

FALLBACK = ("the small machine counted every letter of the paragraph and then began "
    "to write its own strange sentences about the city and the lake and the long "
    "quiet train ride home it wrote about the coach and the counselor and the "
    "quiet gym at seven in the morning and although every line was gibberish the "
    "shape of the words was english because the counts had come from english ") * 8

try:
    raw = urllib.request.urlopen("https://www.gutenberg.org/files/11/11-0.txt", timeout=15).read().decode("utf-8")
    raw = raw[raw.find("Alice was beginning"):raw.find("THE END")]
    print("Loaded Alice in Wonderland:", len(raw), "characters")
except Exception as e:
    raw = FALLBACK
    print("Download failed (%s) - using the built-in backup corpus." % type(e).__name__)

# Keep only lowercase letters and spaces - 27 symbols total.
corpus = re.sub(r"[^a-z ]+", " ", raw.lower())
corpus = re.sub(r" +", " ", corpus).strip()
print("Cleaned corpus:", len(corpus), "characters")
print(repr(corpus[:100]))

In [ ]:
def pair_counts(tokens):
    counts = {}
    for a, b in zip(tokens, tokens[1:]):
        if a == " " or b == " ":
            continue                      # keep merges inside words
        counts[(a, b)] = counts.get((a, b), 0) + 1
    return counts

def merge(tokens, pair):
    a, b = pair
    out, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
            out.append(a + b)
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out

# Learn merges from a slice of the corpus (BPE on the whole book is slow in pure Python).
train_text = corpus[:20000]
tokens = list(train_text)
merges = []
for step in range(60):
    counts = pair_counts(tokens)
    if not counts:
        break
    best = max(counts.items(), key=lambda kv: kv[1])
    if best[1] < 2:
        break
    tokens = merge(tokens, best[0])
    merges.append(best[0])
    if step < 15 or step % 10 == 0:
        print(f"merge {step+1:2d}: {best[0][0]!r} + {best[0][1]!r} -> {best[0][0]+best[0][1]!r}  ({best[1]} times)")

print()
print("vocabulary learned:", [a + b for a, b in merges[:25]], "...")

Nobody told it "the" is a word. Frequency did.

## Tokenize anything

Apply the learned merges, in order, to new text.

In [ ]:
def tokenize(text):
    toks = list(text)
    for pair in merges:
        toks = merge(toks, pair)
    return toks

for sentence in ["the weather in chicago",
                 "strawberry",
                 "your name here"]:          # <- put your actual name in
    toks = tokenize(sentence)
    shown = [t for t in toks if t != " "]
    print(f"{sentence!r} -> {shown}  ({len(shown)} tokens)")

## Turn-in

Tokenize a plain sentence, a sentence with your name, and "strawberry".
For each: how many tokens, and which splits surprised you? One closing
sentence connecting what you saw to a model miscounting letters — the model
sees those chunks, not letters.